## Data Cleaning

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Convert the Excel dataset to CSV format
df = pd.read_excel("First Dataset.xlsx")
df.to_csv("First Dataset.csv", index=False)

In [3]:
# Load the original dataset
df = pd.read_csv("First Dataset.csv")

In [4]:
# Create a copy for data cleaning
clean_df = df.copy()

In [5]:
clean_df.shape

(61, 17)

In [6]:
# Check the data types of all columns
clean_df.dtypes

customer_id             int64
first_name             object
gender                 object
age                   float64
city                   object
province               object
signup_date            object
membership_tier        object
purchase_count          int64
avg_order_value       float64
total_spending        float64
last_purchase_days      int64
payment_method         object
device                 object
discount_used          object
returned_items          int64
satisfaction_score      int64
dtype: object

In [7]:
# Check unique values in the age column
clean_df["age"].unique()

array([ 19.,  53.,  31.,  58.,  28.,  43.,  23.,  61., 145.,  41.,  51.,
        39.,  35.,  47.,  45.,  49.,  30.,  22.,  nan,  50.,  60.,  37.,
        25.,  34.,  63.,  26.,  42.,  65.,  64.,  54.,  32.,  48.,  59.,
        62.,  57.])

In [8]:
# Check for the suspicious age value 145
clean_df[clean_df["age"] ==145]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
9,1010,Arash,F,145.0,Karaj,Alborz,2022-06-25,Silver,19,381.13,7241.47,276,Online Wallet,Android,No,7,1


In [9]:
# Convert age to nullable integer type
clean_df["age"] = clean_df["age"].astype("Int64")

In [10]:
# Replace unrealistic ages (>100) with missing values
clean_df.loc[clean_df["age"] > 100, "age"] = np.nan

In [11]:
# Verify that no unrealistic ages remain
clean_df[clean_df["age"] > 100]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score


In [12]:
# Replace missing ages with the median age and convert the column back to intege
clean_df["age"] = clean_df["age"].fillna(
    clean_df["age"].median()
).astype(int)

In [13]:
clean_df["discount_used"].unique()

array(['Yes', 'No'], dtype=object)

In [14]:
# Convert Yes/No values to Boolean
clean_df["discount_used"] = clean_df["discount_used"].map({
    "Yes": True,
    "No": False
})

In [15]:
clean_df["signup_date"].head(10)

0    2025-02-19
1    2022-08-19
2    2023-06-20
3    2021-11-08
4    2021-10-21
5    2022-01-26
6    2025-09-09
7    2024-10-03
8    2021-05-14
9    2022-06-25
Name: signup_date, dtype: object

In [16]:
# Check whether signup_date values follow the expected YYYY-MM-DD format
clean_df["signup_date"].str.match(r"^\d{4}-\d{2}-\d{2}$").value_counts()

signup_date
True    61
Name: count, dtype: int64

In [17]:
# Convert signup_date to datetime format
clean_df["signup_date"] = pd.to_datetime(clean_df["signup_date"])

In [18]:
clean_df[["age", "discount_used", "signup_date"]].dtypes

age                       int64
discount_used              bool
signup_date      datetime64[ns]
dtype: object

In [19]:
# Check the number of missing values in each column
clean_df.isna().sum()

customer_id           0
first_name            0
gender                0
age                   0
city                  0
province              0
signup_date           0
membership_tier       0
purchase_count        0
avg_order_value       0
total_spending        1
last_purchase_days    0
payment_method        0
device                0
discount_used         0
returned_items        0
satisfaction_score    0
dtype: int64

In [20]:
# Inspect records with missing total_spending
clean_df[clean_df["total_spending"].isna()]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
39,1040,Amir,F,65,Tehran,Tehran,2021-11-02,Bronze,34,37.66,NaN,126,Card,Android,True,3,2


In [21]:
# Calculate expected total spending
calculated = (
    clean_df["purchase_count"] * clean_df["avg_order_value"]
)

# Check whether total_spending is close to the calculated value using a tolerance of 0.01
check = np.isclose(
    clean_df["total_spending"],
    calculated,
    atol=0.01,
    equal_nan=False
)

clean_df[~check]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
29,1030,Sina,M,60,Mashhad,Khorasan,2025-03-07,VIP,26,156.89,25000.0,340,Cash,Web,True,4,4
39,1040,Amir,F,65,Tehran,Tehran,2021-11-02,Bronze,34,37.66,NaN,126,Card,Android,True,3,2


In [22]:
# Flag missing or inconsistent total_spending values, 1 = needs review, 0 = no issue detected
clean_df["total_spending_flag"] = (
    clean_df["total_spending"].isna()
    | ((clean_df["total_spending"] - calculated).abs() > 0.01)
).astype(int)

In [23]:
# Add the calculated value as a suggested value
clean_df["suggested_total_spending"] = calculated.round(2)

In [24]:
# Add the calculated value as a suggested value
clean_df.duplicated().sum()

np.int64(1)

In [25]:
# Display all duplicated rows
clean_df[clean_df.duplicated(keep=False)]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score,total_spending_flag,suggested_total_spending
13,1014,Reza,M,35,Tabriz,East Azerbaijan,2022-08-26,VIP,31,108.19,3353.89,97,Card,Android,False,5,4,0,3353.89
60,1014,Reza,M,35,Tabriz,East Azerbaijan,2022-08-26,VIP,31,108.19,3353.89,97,Card,Android,False,5,4,0,3353.89


In [26]:
# Remove duplicated rows
clean_df = clean_df.drop_duplicates()

In [27]:
clean_df.duplicated().sum()

np.int64(0)

In [28]:
# Define columns containing text/categorical values
text_cols = [
    "first_name",
    "gender",
    "city",
    "province",
    "membership_tier",
    "payment_method",
    "device",
    "discount_used"
]

# Compare the number of unique values before and after normalization
for col in text_cols:
    original = clean_df[col].dropna().astype(str)
    normalized = original.str.strip().str.lower()

    print(
        f"{col}: "
        f"original={original.nunique()}, "
        f"normalized={normalized.nunique()}"
    )

first_name: original=12, normalized=12
gender: original=2, normalized=2
city: original=8, normalized=8
province: original=8, normalized=8
membership_tier: original=4, normalized=4
payment_method: original=3, normalized=3
device: original=3, normalized=3
discount_used: original=2, normalized=2


In [29]:
# Generate descriptive statistics for numeric columns
clean_df[[
    "age",
    "purchase_count",
    "avg_order_value",
    "total_spending",
    "last_purchase_days",
    "returned_items",
    "satisfaction_score"
]].describe()

,age,purchase_count,avg_order_value,total_spending,last_purchase_days,returned_items,satisfaction_score
count,60.000000,60.000000,60.000000,59.000000,60.000000,60.000000,60.000000
mean,43.700000,17.383333,213.156500,3756.375424,198.600000,4.183333,2.983333
std,13.946994,10.149880,130.509923,4298.515563,97.843043,2.683861,1.431979
min,19.000000,0.000000,27.630000,0.000000,3.000000,0.000000,1.000000
25%,31.750000,10.750000,108.137500,1165.330000,134.250000,2.000000,2.000000
50%,45.000000,17.000000,165.435000,2140.380000,203.000000,4.000000,3.000000
75%,58.000000,24.500000,324.107500,4769.250000,273.750000,7.000000,4.000000
max,65.000000,35.000000,449.810000,25000.000000,365.000000,8.000000,5.000000


In [30]:
# Define numeric columns for outlier detection
numeric_cols = [
    "age",
    "purchase_count",
    "avg_order_value",
    "total_spending",
    "last_purchase_days",
    "returned_items",
    "satisfaction_score"
]

# Detect outliers using the IQR method
for col in numeric_cols:
    Q1 = clean_df[col].quantile(0.25)
    Q3 = clean_df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = clean_df[(clean_df[col] < lower) | (clean_df[col] > upper)]
    
    print(f"{col}: {len(outliers)} outlier(s)")

age: 0 outlier(s)
purchase_count: 0 outlier(s)
avg_order_value: 0 outlier(s)
total_spending: 5 outlier(s)
last_purchase_days: 0 outlier(s)
returned_items: 0 outlier(s)
satisfaction_score: 0 outlier(s)


In [31]:
# Calculate the IQR for total_spending
Q1 = clean_df["total_spending"].quantile(0.25)
Q3 = clean_df["total_spending"].quantile(0.75)
IQR = Q3 - Q1

clean_df[clean_df["total_spending"] > Q3 + 1.5 * IQR][
    ["customer_id", "purchase_count", "avg_order_value", "total_spending"]
]

,customer_id,purchase_count,avg_order_value,total_spending
8,1009,34,341.63,11615.42
11,1012,27,434.50,11731.50
29,1030,26,156.89,25000.00
43,1044,33,434.98,14354.34
56,1057,31,436.54,13532.74


In [32]:
# Identify object/text columns
text_cols = clean_df.select_dtypes(include="object").columns

# Check for leading or trailing whitespace in text columns
for col in text_cols:
    count = clean_df[col].astype(str).str.strip().ne(clean_df[col].astype(str)).sum()
    print(f"{col}: {count}")

first_name: 0
gender: 0
city: 0
province: 0
membership_tier: 0
payment_method: 0
device: 0


In [33]:
# Find similar categorical values that may indicate inconsistent entries
from difflib import get_close_matches

cat_cols = [
    "gender",
    "city",
    "province",
    "membership_tier",
    "payment_method",
    "device",
    "discount_used"
]

for col in cat_cols:
    values = clean_df[col].dropna().astype(str).unique()
    found = False

    print(f"\n--- {col} ---")

    for value in values:
        matches = get_close_matches(value, values, n=3, cutoff=0.7)
        matches = [m for m in matches if m != value]

        if matches:
            print(f"{value} -> {matches}")
            found = True

    if not found:
        print("None")


--- gender ---
None

--- city ---
None

--- province ---
None

--- membership_tier ---
None

--- payment_method ---
None

--- device ---
None

--- discount_used ---
None


In [34]:
# Check for positive purchases with zero total spending
clean_df.loc[
    (clean_df["purchase_count"] > 0) & (clean_df["total_spending"] == 0)
]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score,total_spending_flag,suggested_total_spending


In [35]:
# Check for zero purchases with a positive average order value
clean_df.loc[
    (clean_df["purchase_count"] == 0) & (clean_df["avg_order_value"] > 0)
]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score,total_spending_flag,suggested_total_spending
23,1024,Sina,F,53,Ahvaz,Khuzestan,2025-01-27,Gold,0,63.67,0.0,298,Online Wallet,Android,False,2,1,0,0.0


In [36]:
# Check whether returned items exceed the number of purchases
clean_df.loc[
    clean_df["returned_items"] > clean_df["purchase_count"]
]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score,total_spending_flag,suggested_total_spending
7,1008,Reza,F,23,Rasht,Gilan,2024-10-03,VIP,3,389.58,1168.74,195,Online Wallet,iPhone,False,8,1,0,1168.74
14,1015,Kimia,F,47,Ahvaz,Khuzestan,2021-07-24,Gold,3,307.91,923.73,55,Card,Android,True,8,4,0,923.73
23,1024,Sina,F,53,Ahvaz,Khuzestan,2025-01-27,Gold,0,63.67,0.00,298,Online Wallet,Android,False,2,1,0,0.00
28,1029,Ali,F,34,Tabriz,East Azerbaijan,2023-07-03,Gold,2,323.32,646.64,319,Cash,Web,True,4,5,0,646.64
36,1037,Zahra,M,42,Ahvaz,Khuzestan,2024-01-12,Gold,1,387.68,387.68,199,Cash,Web,True,7,2,0,387.68
55,1056,Kimia,M,61,Tabriz,East Azerbaijan,2025-01-25,Silver,1,449.81,449.81,240,Online Wallet,Web,False,4,2,0,449.81


In [37]:
# Flag records requiring review based on purchase and return conditions, 1 = flagged for review, 0 = no issue detected
clean_df["purchase_return_flag"] = (
    (
        (clean_df["purchase_count"] == 0) &
        (clean_df["avg_order_value"] > 0)
    )
    |
    (
        clean_df["returned_items"] > clean_df["purchase_count"]
    )
).astype(int)

In [38]:
# Display records flagged for purchase/return inconsistencies
clean_df.loc[
    clean_df["purchase_return_flag"] == 1,
    [
        "customer_id",
        "purchase_count",
        "avg_order_value",
        "returned_items"
    ]
]

,customer_id,purchase_count,avg_order_value,returned_items
7,1008,3,389.58,8
14,1015,3,307.91,8
23,1024,0,63.67,2
28,1029,2,323.32,4
36,1037,1,387.68,7
55,1056,1,449.81,4


In [39]:
# Check records with no purchases or returns but a recorded satisfaction score
clean_df.loc[
    (clean_df["purchase_count"] == 0) &
    (clean_df["returned_items"] == 0) &
    (clean_df["satisfaction_score"].notna())
]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score,total_spending_flag,suggested_total_spending,purchase_return_flag


In [40]:
# Check for discount usage when no purchases were made
clean_df.loc[
    (clean_df["discount_used"] == "Yes") & (clean_df["purchase_count"] == 0)
]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score,total_spending_flag,suggested_total_spending,purchase_return_flag


In [41]:
# Check for discount usage when the average order value is zero
clean_df.loc[
    (clean_df["discount_used"] == "Yes") & (clean_df["avg_order_value"] == 0)
]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score,total_spending_flag,suggested_total_spending,purchase_return_flag


In [42]:
# Check for discount usage when total spending is zero
clean_df.loc[
    (clean_df["discount_used"] == "Yes") & (clean_df["total_spending"] == 0)
]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score,total_spending_flag,suggested_total_spending,purchase_return_flag


In [43]:
# Check records with no purchases or returns but a recorded last purchase value
clean_df.loc[
    (clean_df["purchase_count"] == 0) &
    (clean_df["returned_items"] == 0) &
    (clean_df["last_purchase_days"].notna())
]

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score,total_spending_flag,suggested_total_spending,purchase_return_flag


In [44]:
# Check for negative values in numeric columns
cols = [
    "age",
    "purchase_count",
    "avg_order_value",
    "total_spending",
    "last_purchase_days",
    "returned_items"
]

for col in cols:
    print(f"{col}: {(clean_df[col] < 0).sum()}")

age: 0
purchase_count: 0
avg_order_value: 0
total_spending: 0
last_purchase_days: 0
returned_items: 0


In [45]:
# Check the latest signup date in the dataset
clean_df["signup_date"].max()

Timestamp('2025-09-09 00:00:00')

In [46]:
# Display names grouped by recorded gender
for gender, group in clean_df.groupby("gender"):
    print(f"\n{gender}:")
    print(group["first_name"].to_list())


F:
['Reza', 'Parsa', 'Sina', 'Amir', 'Reza', 'Reza', 'Arash', 'Neda', 'Arash', 'Neda', 'Kimia', 'Neda', 'Parsa', 'Ali', 'Amir', 'Ali', 'Sina', 'Neda', 'Kimia', 'Mina', 'Ali', 'Neda', 'Neda', 'Sina', 'Amir', 'Sina', 'Maryam', 'Sina', 'Ali', 'Reza', 'Parsa', 'Sara', 'Reza', 'Amir', 'Ali']

M:
['Sina', 'Kimia', 'Kimia', 'Reza', 'Neda', 'Kimia', 'Reza', 'Amir', 'Sina', 'Zahra', 'Maryam', 'Arash', 'Zahra', 'Amir', 'Parsa', 'Neda', 'Kimia', 'Mina', 'Maryam', 'Arash', 'Sina', 'Mina', 'Kimia', 'Maryam', 'Ali']


In [47]:
# Read the reference dataset
gender_ref = pd.read_csv("persian-gender-by-name.csv")

In [48]:
# Inspect the columns of the gender reference dataset
gender_ref.columns

Index(['name', 'gender', 'english_name'], dtype='object')

In [49]:
# Clean names in both datasets
clean_df["name_clean"] = (
    clean_df["first_name"]
    .str.strip()
    .str.lower()
)
gender_ref["name_clean"] = (
    gender_ref["english_name"]
    .str.strip()
    .str.lower()
)

In [50]:
# Standardize gender values in the reference dataset
gender_ref["gender"] = (
    gender_ref["gender"]
    .str.strip()
    .str.upper()
)

In [51]:
# Create a gender reference map
gender_map = (
    gender_ref
    .dropna(subset=["name_clean", "gender"])
    .drop_duplicates("name_clean")
    .set_index("name_clean")["gender"]
)

In [52]:
# Create a suggested gender column
clean_df["suggested_gender"] = (
    clean_df["name_clean"].map(gender_map)
)

In [53]:
# Create a validation table by matching names with the gender reference dataset
check = clean_df.merge(
    gender_ref[["name_clean", "gender"]],
    on="name_clean",
    how="left"
)

In [54]:
# Check the data type and unique gender values in both datasets
print(type(check))
print(check["gender_x"].unique())
print(check["gender_y"].unique())

<class 'pandas.core.frame.DataFrame'>
['F' 'M']
['M' nan 'F']


In [55]:
# Check how Kimia is represented in the dataset
check.loc[
    check["first_name"].str.lower().str.contains("kimia", na=False),
    ["first_name", "name_clean", "gender_x", "gender_y"]
]

,first_name,name_clean,gender_x,gender_y
5,Kimia,kimia,M,NaN
11,Kimia,kimia,M,NaN
24,Kimia,kimia,F,NaN
33,Kimia,kimia,M,NaN
44,Kimia,kimia,F,NaN
70,Kimia,kimia,M,NaN
93,Kimia,kimia,M,NaN


In [56]:
# Check how Kimia is represented in the reference data
gender_ref[
    gender_ref["english_name"]
    .str.lower()
    .str.contains("kimia", na=False)
][["english_name", "gender"]]

,english_name,gender


In [57]:
# Manually add a suggestion for names not available in the reference dataset
clean_df.loc[
    clean_df["name_clean"] == "kimia",
    "suggested_gender"
] = "F"

In [58]:
# Check the suggested gender for Kimia
clean_df.loc[
    clean_df["first_name"].str.strip().str.lower() == "kimia",
    ["first_name", "suggested_gender"]
]

,first_name,suggested_gender
4,Kimia,F
8,Kimia,F
14,Kimia,F
19,Kimia,F
26,Kimia,F
42,Kimia,F
55,Kimia,F


In [59]:
# Flag potential inconsistencies
clean_df["gender_flag"] = (
    clean_df["suggested_gender"].notna() &
    (
        clean_df["gender"].str.upper()
        != clean_df["suggested_gender"].str.upper()
    )
).astype(int)

In [60]:
# Remove temporary name-matching column from the final dataset
clean_df.drop(columns="name_clean", inplace=True)

In [61]:
clean_df.info()
clean_df.describe().T

<class 'pandas.core.frame.DataFrame'>
Index: 60 entries, 0 to 59
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               60 non-null     int64         
 1   first_name                60 non-null     object        
 2   gender                    60 non-null     object        
 3   age                       60 non-null     int64         
 4   city                      60 non-null     object        
 5   province                  60 non-null     object        
 6   signup_date               60 non-null     datetime64[ns]
 7   membership_tier           60 non-null     object        
 8   purchase_count            60 non-null     int64         
 9   avg_order_value           60 non-null     float64       
 10  total_spending            59 non-null     float64       
 11  last_purchase_days        60 non-null     int64         
 12  payment_method            60 

,count,mean,min,25%,50%,75%,max,std
customer_id,60.0,1030.5,1001.0,1015.75,1030.5,1045.25,1060.0,17.464249
age,60.0,43.7,19.0,31.75,45.0,58.0,65.0,13.946994
signup_date,60,2023-02-03 14:00:00,2021-02-23 00:00:00,2021-12-06 12:00:00,2022-08-15 00:00:00,2024-06-14 12:00:00,2025-09-09 00:00:00,NaN
purchase_count,60.0,17.383333,0.0,10.75,17.0,24.5,35.0,10.14988
avg_order_value,60.0,213.1565,27.63,108.1375,165.435,324.1075,449.81,130.509923
total_spending,59.0,3756.375424,0.0,1165.33,2140.38,4769.25,25000.0,4298.515563
last_purchase_days,60.0,198.6,3.0,134.25,203.0,273.75,365.0,97.843043
returned_items,60.0,4.183333,0.0,2.0,4.0,7.0,8.0,2.683861
satisfaction_score,60.0,2.983333,1.0,2.0,3.0,4.0,5.0,1.431979
total_spending_flag,60.0,0.033333,0.0,0.0,0.0,0.0,1.0,0.18102


In [62]:
clean_df.columns.tolist()

['customer_id',
 'first_name',
 'gender',
 'age',
 'city',
 'province',
 'signup_date',
 'membership_tier',
 'purchase_count',
 'avg_order_value',
 'total_spending',
 'last_purchase_days',
 'payment_method',
 'device',
 'discount_used',
 'returned_items',
 'satisfaction_score',
 'total_spending_flag',
 'suggested_total_spending',
 'purchase_return_flag',
 'suggested_gender',
 'gender_flag']

In [63]:
clean_df.to_excel("cleaned_dataset.xlsx", index=False)